In [ ]:
!nvidia-smi

Fri Oct  2 02:44:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"))
print(whoami()["name"])

txchrixo


In [1]:
!pip install -q -U "transformers>=4.57" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 115.5 MB/s eta 0:00:00


In [1]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-1.5-4b-it"
# MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

a = torch.randn(256, 256, device="cuda", dtype=torch.bfloat16)
print("bfloat16 ok:", torch.isfinite((a @ a).float()).all().item())

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(torch.cuda.get_device_name(0), f"{torch.cuda.memory_allocated() / 1e9:.1f} GB used")

bfloat16 ok: True


config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Tesla T4 8.6 GB used


In [ ]:
import time

def ask(text, image=None, max_new_tokens=1500):
    content = [{"type": "text", "text": text}]
    if image is not None:
        content.append({"type": "image", "image": image})
    inputs = processor.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True,
                                           tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
    n = inputs["input_ids"].shape[-1]
    t = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    print(f"({time.time() - t:.0f} s, {len(out[0]) - n} tokens)")
    return processor.decode(out[0][n:], skip_special_tokens=True).strip()

print(ask("Réponds en une phrase : quelle est la capitale du Cameroun ?"))

(3 s, 11 tokens)
La capitale du Cameroun est Yaoundé.


In [3]:
import io
from google.colab import files
from PIL import Image

uploaded = files.upload()
images = {name: Image.open(io.BytesIO(data)).convert("RGB") for name, data in uploaded.items()}
print(list(images))

Saving R001_base.png to R001_base.png
['R001_base.png']


In [4]:
QUESTION = "Voici le résultat d'analyse de mon oncle. Qu'est-ce que ça veut dire ? Est-ce que c'est grave ?"

for name, img in images.items():
    print("=" * 80)
    print(name)
    print(ask(QUESTION, img))

R001_base.png
(46 s, 404 tokens)
Bonjour,

Je ne peux pas interpréter directement les résultats d'analyse pour vous. L'interprétation des résultats d'analyse doit être faite par un professionnel de la santé (médecin, pharmacien, etc.) qui connaît votre dossier médical complet.

Cependant, je peux vous donner quelques informations générales sur ce type de résultat :

*   **PSA total :** Le PSA (Antigène prostatique spécifique) est une protéine produite par les cellules de la prostate. Son taux peut être élevé pour diverses raisons, dont certaines sont bénignes (comme une hypertrophie bénigne de la prostate) et d'autres peuvent être liées à des problèmes plus graves (comme un cancer de la prostate).
*   **Résultat de 3,80 ng/mL :** Le seuil de référence est généralement inférieur à 4,00 ng/mL. Votre résultat est donc légèrement supérieur à ce seuil.

**Ce que cela signifie :**

Votre résultat de PSA est légèrement élevé par rapport au seuil de référence. Cela ne signifie pas automatiquem

In [5]:
name = list(images)[0]
print(ask("What is the PSA value on this report, with its unit?", images[name]))

(6 s, 28 tokens)
Based on the image provided, the PSA (Prostate-Specific Antigen) value is **3.80 ng/mL**.
